In [1]:
import numpy as np
import pandas as pd

In [5]:
protein_data = {
    'protein_id': 'P04637',
    'sequence': 'MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP...',
    'true_labels': {
        'MFO': ['GO:0003677', 'GO:0043565', 'GO:0000981', 'GO:0000978', 
                'GO:0001228', 'GO:0008134', 'GO:0005515'],
        'BPO': ['GO:0000122', 'GO:0006357', 'GO:0045893', 'GO:0006974',
                'GO:0006915', 'GO:0007049', 'GO:0030308', 'GO:0043066', 'GO:2000045'],
        'CCO': ['GO:0005634', 'GO:0005654', 'GO:0005737', 'GO:0005829']
    }
}

In [7]:
go_terms = [
    # MFO terms (10 terms)
    'GO:0003677',  # DNA binding
    'GO:0043565',  # sequence-specific DNA binding  
    'GO:0000981',  # RNA polymerase II transcription factor activity
    'GO:0003674',  # molecular function (root)
    'GO:0003824',  # catalytic activity
    'GO:0005488',  # binding
    'GO:0000166',  # nucleotide binding
    'GO:0005515',  # protein binding
    'GO:0005524',  # ATP binding
    'GO:0003700',  # transcription factor activity
    
    # BPO terms (5 terms)
    'GO:0006357',  # regulation of transcription
    'GO:0006915',  # apoptotic process
    'GO:0007049',  # cell cycle
    'GO:0006974',  # response to DNA damage
    'GO:0008150',  # biological process (root)
    
    # CCO terms (5 terms)
    'GO:0005634',  # nucleus
    'GO:0005737',  # cytoplasm
    'GO:0005623',  # cell (root)
    'GO:0005886',  # plasma membrane
    'GO:0005829',  # cytosol
]

In [8]:
# True labels dalam format binary vector (20 dimensi untuk contoh ini)
true_labels_binary = np.array([
    1, 1, 1, 1, 0, 1, 0, 1, 0, 1,  # MFO: 10 terms
    1, 1, 1, 1, 1,                # BPO: 5 terms  
    1, 1, 1, 0, 1                 # CCO: 5 terms
])

In [9]:
# Prediksi model (confidence scores antara 0-1)
predicted_scores = np.array([
    0.95, 0.92, 0.88, 0.30, 0.05, 0.70, 0.10, 0.85, 0.03, 0.90,  # MFO
    0.96, 0.94, 0.89, 0.91, 0.20,                              # BPO
    0.97, 0.82, 0.25, 0.08, 0.75                               # CCO
])

In [10]:
# Threshold untuk konversi scores ke binary predictions
threshold = 0.5
predicted_binary = (predicted_scores >= threshold).astype(int)

In [11]:
print("=== MULTI-LABEL CLASSIFICATION EXAMPLE ===")
print(f"Protein: {protein_data['protein_id']}")
print(f"Total GO terms dalam contoh: {len(go_terms)}")
print(f"True labels aktif: {sum(true_labels_binary)}")
print(f"Predicted labels aktif (threshold={threshold}): {sum(predicted_binary)}")
print()

=== MULTI-LABEL CLASSIFICATION EXAMPLE ===
Protein: P04637
Total GO terms dalam contoh: 20
True labels aktif: 16
Predicted labels aktif (threshold=0.5): 13



In [13]:
# Tampilkan hasil per ontology
def print_predictions(ontology, start_idx, end_idx):
    print(f"\n{ontology} Predictions:")
    print("-" * 60)
    for i in range(start_idx, end_idx):
        term = go_terms[i]
        true = "✓" if true_labels_binary[i] else "✗"
        pred = "✓" if predicted_binary[i] else "✗"
        score = predicted_scores[i]
        
        status = "CORRECT" if true_labels_binary[i] == predicted_binary[i] else "WRONG"
        print(f"{term:35} True:{true} Pred:{pred} ({score:.2f}) [{status}]")

print_predictions("Molecular Function (MFO)", 0, 10)
print_predictions("Biological Process (BPO)", 10, 15)
print_predictions("Cellular Component (CCO)", 15, 20)


Molecular Function (MFO) Predictions:
------------------------------------------------------------
GO:0003677                          True:✓ Pred:✓ (0.95) [CORRECT]
GO:0043565                          True:✓ Pred:✓ (0.92) [CORRECT]
GO:0000981                          True:✓ Pred:✓ (0.88) [CORRECT]
GO:0003674                          True:✓ Pred:✗ (0.30) [WRONG]
GO:0003824                          True:✗ Pred:✗ (0.05) [CORRECT]
GO:0005488                          True:✓ Pred:✓ (0.70) [CORRECT]
GO:0000166                          True:✗ Pred:✗ (0.10) [CORRECT]
GO:0005515                          True:✓ Pred:✓ (0.85) [CORRECT]
GO:0005524                          True:✗ Pred:✗ (0.03) [CORRECT]
GO:0003700                          True:✓ Pred:✓ (0.90) [CORRECT]

Biological Process (BPO) Predictions:
------------------------------------------------------------
GO:0006357                          True:✓ Pred:✓ (0.96) [CORRECT]
GO:0006915                          True:✓ Pred:✓ (0.94) [CORRECT

In [15]:
# Hitung metrik
TP = np.sum((predicted_binary == 1) & (true_labels_binary == 1))
FP = np.sum((predicted_binary == 1) & (true_labels_binary == 0))
FN = np.sum((predicted_binary == 0) & (true_labels_binary == 1))

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"\n=== PERFORMANCE METRICS ===")
print(f"Precision: {precision:.3f} ({TP}/{TP+FP} correct predictions)")
print(f"Recall:    {recall:.3f} ({TP}/{TP+FN} true labels captured)")
print(f"F1-score:  {f1:.3f}")


=== PERFORMANCE METRICS ===
Precision: 1.000 (13/13 correct predictions)
Recall:    0.812 (13/16 true labels captured)
F1-score:  0.897


In [18]:
train_taxon = pd.read_csv('../data/Train/train_taxonomy.tsv')
train_taxon.head()

,A0A0C5B5G6\t9606
0,A0JNW5\t9606
1,A0JP26\t9606
2,A0PK11\t9606
3,A1A4S6\t9606
4,A1A519\t9606
